# DATALUS Training Factory: Massive Scale & Multi-GPU

This notebook demonstrates the **DATALUS Training Factory**, designed for heavy compute environments (e.g., Kaggle 2x T4 or AWS G4 instances). 
Unlike the POC, we focus strictly on massive data ingestion, Out-Of-Memory (OOM) safe Polars processing, 
Distributed Multi-GPU training using `torch.nn.DataParallel`, and Edge deployment export via Post-Training Quantization (PTQ).

The objective is to demonstrate the framework's scalability when handling millions of records from the Brazilian Health Information System (SIH-SUS).

## 1. Environment Setup
Dynamically detecting the environment (Kaggle, Colab, Local) and installing required packages.

In [ ]:
import os
import sys
from pathlib import Path

def get_working_dir():
    if 'google.colab' in sys.modules:
        return '/content/datalus_factory'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return '/kaggle/working/datalus_factory'
    return './datalus_factory'

WORKING_DIR = get_working_dir()
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Working directory: {WORKING_DIR}')

INSTALLED = Path(WORKING_DIR) / '.installed'

if not INSTALLED.exists():
    if 'google.colab' in sys.modules or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        !git clone --depth 1 https://github.com/emanuellcs/datalus.git "{WORKING_DIR}/source" 2>/dev/null || true
        %cd "{WORKING_DIR}/source"
        !pip install --quiet -e '.[dev]' pysus polars nest_asyncio 2>&1 | tail -5
        sys.path.insert(0, f'{WORKING_DIR}/source')
        INSTALLED.touch()
        print()
        print('=' * 60)
        print('  INSTALLATION COMPLETE')
        print('  Click "RESTART RUNTIME" in the warning above,')
        print('  or go to Runtime > Restart and run all')
        print('=' * 60)
        os._exit(0)
    else:
        INSTALLED.touch()
        print('Local environment. Dependencies assumed installed.')
else:
    src = Path(WORKING_DIR) / 'source'
    if src.is_dir():
        sys.path.insert(0, str(src))
    print('Dependencies already installed.')


## 2. Heavy Data Ingestion (SIH-SUS 2023 Full Year)
We simulate real-world scale by downloading an entire year of SIH-SUS data (São Paulo, 2023) using the `pysus` API. 
Saving the concatenated chunks to a single massive Parquet file allows the DATALUS Polars engine to utilize **lazy evaluation** (`pl.scan_parquet`). 

This architectural choice guarantees that we avoid RAM exhaustion even when processing datasets that exceed available memory, as data is streamed efficiently to the tensor encoding layer.

In [ ]:
from pysus import sih, sim, sinan
import polars as pl
import numpy as np
from datetime import datetime
from pathlib import Path
import urllib.request
import asyncio

print('Downloading clinical data from DATASUS...')

STATES = ['SP', 'RJ', 'MG', 'PR', 'RS', 'BA', 'DF']

def fmt_tier(label, state, year, month, count):
    return f'Tier {label}: {state}/{year}/{month} ({count} records)'

def try_pysus():
    now = datetime.now()
    sy, sm = (now.year, now.month - 3) if now.month > 3 else (now.year - 1, now.month + 9)
    for state in STATES:
        for y in range(sy, sy - 2, -1):
            for m in range(sm if y == sy else 12, 0, -1):
                try:
                    df = sih(state=state, year=y, month=[m], group='RD', as_dataframe=True)
                    if df is not None and len(df) > 0 and 'MORTE' in df.columns:
                        print(fmt_tier('1: SIH-RD', state, y, m, len(df)))
                        return pl.from_pandas(df)
                except Exception:
                    continue
        for y in range(sy, sy - 2, -1):
            try:
                df = sim(state=state, year=y, as_dataframe=True)
                if df is not None and len(df) > 0:
                    print(fmt_tier('2: SIM', state, y, 'all', len(df)))
                    df['MORTE'] = 1
                    return pl.from_pandas(df)
            except Exception:
                continue
    for y in range(sy, sy - 2, -1):
        try:
            df = sinan(disease='deng', year=y, as_dataframe=True)
            if df is not None and len(df) > 0:
                print(fmt_tier('3: SINAN-deng', 'BR', y, 'all', len(df)))
                if 'MORTE' not in df.columns:
                    df['MORTE'] = 0
                return pl.from_pandas(df)
        except Exception:
            continue
    return None

def try_direct_ftp():
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        return None
    from pysus.api.extensions import DBC
    now = datetime.now()
    sy, sm = (now.year, now.month - 3) if now.month > 3 else (now.year - 1, now.month + 9)
    base = 'ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/199301_/Dados/'
    for state in ['SP', 'RJ', 'MG']:
        for y in range(sy, sy - 2, -1):
            for m in range(sm if y == sy else 12, 0, -1):
                fname = f'RD{state}{y % 100:02d}{m:02d}.dbc'
                try:
                    with urllib.request.urlopen(base + fname, timeout=30) as r:
                        dbc_path = Path('/tmp') / fname
                        dbc_path.write_bytes(r.read())
                    df = asyncio.run(DBC(path=dbc_path).load())
                    if df is not None and len(df) > 0:
                        print(fmt_tier('4: FTP direct', state, y, m, len(df)))
                        return pl.from_pandas(df)
                except Exception:
                    continue
    return None

def generate_sample():
    print('Tier 5: Generated demo sample (real data unavailable)')
    print('NOTE: This is not real data — results are for pipeline demo only.')
    np.random.seed(42)
    n = 1000
    return pl.DataFrame({
        'MORTE': np.random.choice([0, 1], n, p=[0.92, 0.08]).astype(np.int64),
        'IDADE': np.random.randint(0, 100, n).astype(np.float64),
        'SEXO': np.random.choice(['0', '1'], n),
        'DIAS_PERMANENCIA': np.clip(np.random.poisson(5, n), 0, 100).astype(np.float64),
        'VALOR_TOTAL': np.round(np.random.lognormal(7, 1.5, n), 2),
        'PROCEDIMENTO_PRINCIPAL': np.random.choice(['AB', 'CD', 'EF', 'GH', 'IJ'], n),
        'MUNICIPIO_MOV': np.random.choice(['3550308', '3304557', '3106200'], n),
    })

df_massive = try_pysus()
if df_massive is None:
    df_massive = try_direct_ftp()
if df_massive is None:
    df_massive = generate_sample()

df_massive = df_massive.with_columns(
    pl.col('MORTE').cast(pl.Int64).fill_null(0)
)

cols_to_keep = [
    c for c in df_massive.columns
    if not c.startswith(('N_AIH', 'IDENT', 'CNS', 'CPF', 'CNPJ'))
]
df_massive = df_massive.select(
    ['MORTE'] + [c for c in cols_to_keep if c != 'MORTE']
)

raw_path = f'{WORKING_DIR}/massive_raw_sih.parquet'
df_massive.write_parquet(raw_path)
print(f'Massive dataset saved to {raw_path} | Shape: {df_massive.shape}')
print(f'Columns ({len(df_massive.columns)}): {df_massive.columns[:8]}...')

## 3. High-Throughput Ingestion
The `ingest` command maps the raw massive Parquet to the tensor-ready format, handling categorical frequencies and numeric quantile transforms.

In [ ]:
!datalus ingest {WORKING_DIR}/massive_raw_sih.parquet {WORKING_DIR}/processed.parquet --schema-path {WORKING_DIR}/schema_config.json --target-column MORTE

## 4. Distributed Multi-GPU Training Factory
This is the core compute phase. We pass `--gpu "0,1"` to distribute the load across multiple GPUs. 
DATALUS automatically masks environment variables and wraps the residual MLP denoiser in `torch.nn.DataParallel` for scalable throughput.

**Key Components:**
- **Cosine Schedule:** Sophisticated diffusion variance control.
- **EMA Tracking:** Exponential Moving Average of weights to ensure structural stability.
- **Mixed Precision (AMP):** Faster training with lower VRAM footprint.
- **Deterministic Checkpointing:** Resumable training states across the distributed cluster.

In [ ]:
!datalus train {WORKING_DIR}/schema_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/artifacts --epochs 50 --batch-size 2048 --checkpoint-every-steps 1000 --gpu "0,1"

## 5. Artifact Decoupling: Edge Export
After training on high-end hardware, we decouple the generative model. 
Using Post-Training Quantization (**INT8 PTQ**), we compress the massive float32 diffusion model.

This allows the model, despite being trained on 2x T4 GPUs, to run smoothly on local CPUs or directly in the browser via ONNX Runtime Web.

In [ ]:
!datalus export-onnx {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/artifacts --quantize